In [1]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.mixture import unifac
from src.data import load_unifac_data, check_unifac_parameters, check_pure_gc_parameters
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# step1: define the compounds
mixture = {#'acetone': {'CH3':1, 'CH3CO':1},
           'methanol': {'CH3OH':1},
           #'water': {'H2O':1},
           #'DOH': {'DOH':1},
           'methyl_acetate': {'CH3':1, 'CH3COO':1},
           #'water': {'H2O':1},
           #'DMSO': {'DMSO':1}
           #'E': {'CH3':2, 'CH2':1, 'C':1, 'CH3CO':2}
           #'E': {'CH3':4, 'C':1, 'CH2COO':1}
           #'heptane': {'CH3':2, 'CH2':5}
           }
# validate the structures
validator = StructuralValidator()
validator.validate(mixture)

{'methanol': True, 'methyl_acetate': True}

In [3]:
# step2: calculate the required properties
prop_list = ['tc', 'pc', 'tb', 'ac', 'hv']
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)
# fix the properties of acetone
#pure_props['acetone']['tc'] = 508.1
#pure_props['acetone']['pc'] = 48.0
#pure_props['acetone']['tb'] = 329.3
pure_props['methyl_acetate']['tc'] = 510.0     # K
pure_props['methyl_acetate']['pc'] = 47.5      # bar
pure_props['methyl_acetate']['tb'] = 330.0
# pure_props['water']['tc'] = 647.1     # K
# pure_props['water']['pc'] = 220.64    # bar
# pure_props['water']['tb'] = 373.15
pure_props

{'methanol': {'tc': 513.0000000000005,
  'pc': 80.99999999999997,
  'tb': 337.84999999999985,
  'ac': 0.5639909999999987,
  'hv': 37.466699999999996},
 'methyl_acetate': {'tc': 510.0,
  'pc': 47.5,
  'tb': 330.0,
  'ac': 0.3671361376022212,
  'hv': 33.653999999999996}}

In [4]:
# check if unifac dat are available
check_unifac_parameters(mixture)
# Load all UNIFAC parameters
rk_data, qk_data, interaction_data = load_unifac_data()

In [5]:
# step4: perform the LLE

# define number of data points
n_data = 100
# set composition
# generate all binary combinations for ternary system
x0 = 0.0
xf = 1.0

# Binary mixture: x1-x2
x1 = np.linspace(x0, xf, n_data)
x2 = 1 - x1

# Composition array
x = np.column_stack([x1, x2])

# generate temperature matrix
tb_values = [pure_props[compound]['tb'] for compound in mixture.keys()]

T_step = 1.0
T_range = np.arange(min(tb_values), max(tb_values), T_step)
T = np.tile(T_range[:, np.newaxis], (1, n_data))

In [6]:
miscibility_results = []
for i in range(T.shape[0]):
    results = unifac(mixture, x, T[i,:], rk_data, qk_data, interaction_data)
    gamma = results['gamma']
    # Gibbs mixing energy
    G_mix = (np.sum(x * np.log(gamma), axis=1) + 
         np.sum(np.ma.log(x).filled(0) * x, axis=1))
    
    # Second derivative test
    G2_mix = np.diff(np.diff(G_mix))
    
    # Miscibility checks (excluding endpoints with [1:-1])
    is_miscible = np.all(G_mix[1:-1] <= 0) and np.all(G2_mix >= 0)
    
    miscibility_results.append({
        'T': T,
        'miscible': is_miscible,
        'G_mix': G_mix,
        'G2_mix': G2_mix
    })
    #print(f"T = {T[i,0]:.2f} K: Miscible = {is_miscible}")
    if not is_miscible:
        print(f"  T = {T[i,0]:.2f} K:")
        #print(f"  - G_mix all <= 0: {np.all(G_mix[1:-1] <= 0)}")
        #print(f"  - G2_mix all >= 0: {np.all(G2_mix >= 0)}")

In [7]:

miscibility_results[0]['G_mix']

array([ 0.        , -0.04637707, -0.07888083, -0.10622666, -0.13026537,
       -0.15187475, -0.17157092, -0.18969412, -0.20648552, -0.22212488,
       -0.23675119, -0.25047504, -0.26338637, -0.2755597 , -0.28705767,
       -0.29793358, -0.30823331, -0.31799666, -0.32725844, -0.33604931,
       -0.34439641, -0.35232389, -0.35985336, -0.36700419, -0.37379381,
       -0.38023796, -0.38635089, -0.39214549, -0.39763349, -0.40282551,
       -0.40773123, -0.41235946, -0.41671819, -0.42081467, -0.42465551,
       -0.42824667, -0.43159354, -0.43470097, -0.43757331, -0.44021442,
       -0.44262774, -0.44481626, -0.44678258, -0.44852889, -0.45005702,
       -0.45136846, -0.45246431, -0.45334534, -0.45401201, -0.45446441,
       -0.45470233, -0.45472522, -0.45453219, -0.45412206, -0.45349327,
       -0.45264396, -0.45157191, -0.45027453, -0.4487489 , -0.44699169,
       -0.44499919, -0.44276729, -0.44029143, -0.4375666 , -0.43458732,
       -0.43134759, -0.42784086, -0.42406   , -0.41999723, -0.41